# Aircraft Activity

Historical aircraft frequency, identity, type, manufacturer, and operator analysis.

In [ ]:
# Load shared path, database, export, and report-header helpers.
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb
%run report-header.ipynb

# Configure optional spreadsheet and chart exports without hard-coded paths.
export_outputs = False
export_folder = get_export_folder_path()


In [ ]:
# Display the standard report metadata before the report body.
report_metadata = display_report_header('Aircraft Activity')


In [ ]:
# Load the all-session aircraft activity query and normalise date columns.
query = construct_query('tracker', 'reports', 'aircraft-activity.sql', {})
aircraft = query_data('tracker', query)
aircraft['First Observation'] = pd.to_datetime(aircraft['First Observation'])
aircraft['Most Recent Observation'] = pd.to_datetime(aircraft['Most Recent Observation'])
aircraft.head(25)


In [ ]:
# Show the most frequently observed registrations and aircraft addresses.
top_aircraft = aircraft.nlargest(20, 'Observations')
top_aircraft[['Address', 'Registration', 'Observations', 'Sessions', 'First Observation', 'Most Recent Observation']]


In [ ]:
# Aggregate categorical reference data, retaining Unknown values explicitly.
type_summary = aircraft.groupby('Aircraft Type', as_index=False)['Observations'].sum().nlargest(15, 'Observations')
manufacturer_summary = aircraft.groupby('Manufacturer', as_index=False)['Observations'].sum().nlargest(15, 'Observations')
operator_summary = aircraft.groupby('Operator', as_index=False)['Observations'].sum().nlargest(15, 'Observations')
identification_summary = aircraft.groupby('Identified', as_index=False)['Address'].count().rename(columns={'Address':'Aircraft'})

# Use aligned bar charts for quick comparison across the main classifications.
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
top_aircraft.sort_values('Observations').plot.barh(ax=axes[0,0], x='Registration', y='Observations', title='Most frequently observed', legend=False)
type_summary.sort_values('Observations').plot.barh(ax=axes[0,1], x='Aircraft Type', y='Observations', title='By aircraft type', legend=False)
manufacturer_summary.sort_values('Observations').plot.barh(ax=axes[1,0], x='Manufacturer', y='Observations', title='By manufacturer', legend=False)
operator_summary.sort_values('Observations').plot.barh(ax=axes[1,1], x='Operator', y='Observations', title='By operator', legend=False)
plt.tight_layout()
if export_outputs:
    export_chart(export_folder, 'aircraft-activity', 'png')
    export_to_spreadsheet(export_folder, 'aircraft-activity.xlsx', {'Aircraft': aircraft, 'Types': type_summary, 'Manufacturers': manufacturer_summary, 'Operators': operator_summary, 'Identification': identification_summary})


In [ ]:
# Present identified versus unidentified aircraft as a compact table.
identification_summary
